In [ ]:
pip install -qU langchain-openai langchain-core langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.1/420.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.0 MB/s eta 0:00:00


In [ ]:
# ChatOpenAI: This is the langchain wrapper for OpenAI chat models
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnableSequence, RunnableParallel, RunnableBranch
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# Initialize OpenAI chat model
chat_model = ChatOpenAI(
    model_name = "gpt-4o-mini",
    temperature = .5
)

In [ ]:
classification_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content= """You are an AI that classifies text into 'question', 'story', 'statement'.
    only return one of these three words."""),
    ("human", "Classify the following text: \n\n {text} \n Category:")
])

In [ ]:
classification_step = classification_prompt | chat_model | (lambda x: {"category": x.content.strip().lower()})

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content= "You are an AI that answers questions concisely."),
    ("human", "Answer this question: {text}")
])

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content= "You are an AI that summarizes stories concisely."),
    ("human", "summarize this story: {text}")
])

In [ ]:
title_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content= "You are an AI that generates creatitve and engaging titles for statements."),
    ("human", "create a short, engaging title for the following statement: {text}")
])

In [ ]:
answer_question = answer_prompt | chat_model | (lambda x: {"response": x.content.strip()})

In [ ]:
summarize_story = summary_prompt | chat_model | (lambda x: {"response": x.content.strip()})

In [ ]:
generate_title = title_prompt | chat_model | (lambda x: {"response": x.content.strip()})

In [ ]:
pipeline = (
    RunnableLambda(lambda x: {"text": x}) |
    (lambda x: {**x, **classification_step.invoke(x)}) |
    RunnableBranch(
        (lambda x: x["category"] == "question", answer_question),
        (lambda x: x["category"] == "story", summarize_story),
        (generate_title)
    )
)

In [ ]:
inputs = [
    "What is the capital of Japan?",
    "Once upon a time in a faraway land, there was a brave knight who fought dragons to protect his kingdom.",
    "Technology is evolving rapidly, and AI is shaping the future of work.",
    "Self-driving cars will change the way we travel.",
]

In [ ]:
# **Run the pipeline for different inputs**
for input_text in inputs:
    response = pipeline.invoke(input_text)
    print(f"📝 **Input**: {input_text}\n📢 **Response**: {response['response']}\n{'-'*50}")

📝 **Input**: What is the capital of Japan?
📢 **Response**: The capital of Japan is Tokyo.
--------------------------------------------------
📝 **Input**: Once upon a time in a faraway land, there was a brave knight who fought dragons to protect his kingdom.
📢 **Response**: In a distant land, a courageous knight battled dragons to safeguard his kingdom.
--------------------------------------------------
📝 **Input**: Technology is evolving rapidly, and AI is shaping the future of work.
📢 **Response**: "AI Revolution: Shaping Tomorrow's Workforce Today!"
--------------------------------------------------
📝 **Input**: Self-driving cars will change the way we travel.
📢 **Response**: "Revolution on Wheels: The Future of Travel with Self-Driving Cars"
--------------------------------------------------
